In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.regression import *
from xgboost import XGBRegressor
from pathlib import Path

In [2]:
PATH = Path('datasets')
df = pd.read_csv(PATH/"IndyCar_dataset_v25.csv")

In [3]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values("EventDate")

In [4]:
print(df[["DriverID", "NormalizedPositionFinish", "DRFAvg"]].groupby("DriverID").head(3).head(30))

    DriverID  NormalizedPositionFinish  DRFAvg
0       3608                      0.52     NaN
25      4407                      0.64     NaN
24      4401                      0.88     NaN
23      4276                      0.20     NaN
22      4236                      1.00     NaN
21      4216                      0.12     NaN
20      4215                      0.40     NaN
19      4144                      0.32     NaN
17      3813                      0.92     NaN
16      3811                      0.84     NaN
15      3736                      0.72     NaN
14      3682                      0.36     NaN
13      3680                      0.28     NaN
18      4021                      0.80     NaN
11      3672                      0.60     NaN
12      3675                      0.56     NaN
2       3620                      0.68     NaN
3       3622                      0.00     NaN
4       3625                      0.76     NaN
5       3628                      0.04     NaN
1       3616 

In [5]:
df.head()

,DriverName,DriverID,Rookie,DRFAvg,DTAvg,DTTAvg,DNFRate,TDNFRate,DriverElo,DriverTElo,...,BestLapSpeed,LapConsistency,PaceDeg,AvgPaceVSLeadPace,AvgPitTime,PitAvgTimeVSOverall,TrackAvgPitStops,FuelWindowEstimate,PositionFinish,NormalizedPositionFinish
0,Marco Andretti,3608,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,0.52
25,Rubens Barrichello,4407,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,0.64
24,Katherine Legge,4401,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0.88
23,Simon Pagenaud,4276,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,0.20
22,James Jakes,4236,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,1.00


In [ ]:
#Pre-Qualy Non-ID Model
drop_cols = [
    "DriverName", "DriverID", "PositionStart", "TeamName", "TeamID", "CarEngine", "EngineID", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [6]:
#Pre-Qualy ID Model
drop_cols = [
    "DriverName", "PositionStart", "StartVsField","TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

DriverElo                      -3.935091e-01
DriverTTElo                    -3.688284e-01
TeamElo                        -3.252843e-01
DriverTElo                     -2.411139e-01
TeamTElo                       -2.389058e-01
TeamID                         -1.191421e-01
EngineTTElo                    -7.213333e-02
EngineElo                      -5.976131e-02
EngineTElo                     -4.367183e-02
TeamTrackAvgPitStops           -1.353400e-02
TeamTrackAvgPittedCautionPCT   -8.203800e-03
TrackAvgPitStops               -3.352375e-03
TrackID                        -2.446460e-03
TrackAvgSpeed                  -1.585282e-04
FuelWindowEstimate             -1.388490e-06
EraID                          -2.990131e-15
FieldSize                      -1.204454e-15
TrackLength                    -5.211413e-16
TotalRaceLaps                   1.161062e-16
EventTrackTypeID                3.229748e-16
TrackTypeAvgCautionLaps         2.099097e-04
TrackAvgCautionLaps             4.564248e-04
TrackTypeA

In [ ]:
#Post-Qualy Model
drop_cols = [
    "DriverName", "TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = data.fillna(data.median(numeric_only=True))
data_unseen = data_unseen.fillna(data_unseen.median(numeric_only=True))
#data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
#data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [7]:
df = df.drop(columns=drop_cols)

In [8]:
print(df.columns.tolist())

['DriverID', 'Rookie', 'DRFAvg', 'DTAvg', 'DTTAvg', 'DNFRate', 'TDNFRate', 'DriverElo', 'DriverTElo', 'DriverTTElo', 'DriverRitmo', 'TeamID', 'TRP', 'TTP', 'TeamDNFRate', 'TeamElo', 'TeamTElo', 'TeamRitmo', 'TeamTrackAvgPitStops', 'TeamTrackAvgPittedCautionPCT', 'EngineID', 'EngineElo', 'EngineTElo', 'EngineTTElo', 'TrackID', 'EventTrackTypeID', 'TrackAvgCautions', 'TrackAvgCautionLaps', 'TrackTypeAvgCautions', 'TrackTypeAvgCautionLaps', 'TrackAvgSpeed', 'EraID', 'FieldSize', 'TotalRaceLaps', 'TrackLength', 'TrackAvgPitStops', 'FuelWindowEstimate', 'NormalizedPositionFinish']


In [9]:
exp = setup(
    data=data, 
    target="NormalizedPositionFinish", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,NormalizedPositionFinish
2,Target type,Regression
3,Original data shape,"(5608, 38)"
4,Transformed data shape,"(5608, 38)"
5,Transformed train set shape,"(3925, 38)"
6,Transformed test set shape,"(1683, 38)"
7,Numeric features,37
8,Rows with missing values,32.9%
9,Preprocess,True


In [ ]:
compare_models()

In [ ]:
rf = create_model('rf')
rf_tune = tune_model(rf)

In [ ]:
predict_model(rf_tune);
predict_model(rf);
newpred2 = predict_model(rf_tune, data=data_unseen)
newpred2 = predict_model(rf, data=data_unseen)

In [ ]:
gbr = create_model('gbr')

In [ ]:
gbr_tune = tune_model(gbr)

In [ ]:
predict_model(gbr);
#predict_model(gbr_tune);

In [ ]:
#newpred3 = predict_model(gbr_tune, data=data_unseen)
newpred3 = predict_model(gbr, data=data_unseen)

In [10]:
cat = create_model('catboost')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2645,0.0968,0.3111,-0.0665,0.2140,1.2117
1,0.2614,0.0948,0.3080,-0.0473,0.2125,1.0793
2,0.2479,0.0876,0.2959,0.0299,0.2037,1.1060
3,0.2386,0.0851,0.2918,0.0625,0.2015,1.0394
4,0.2482,0.0874,0.2957,0.0420,0.2017,1.0224
5,0.2325,0.0801,0.2830,0.1161,0.1955,1.0522
6,0.2312,0.0779,0.2792,0.1427,0.1889,0.8485
7,0.2385,0.0827,0.2875,0.0879,0.1946,0.9423
8,0.2339,0.0784,0.2799,0.1364,0.1911,1.0345


In [11]:
cat_tune = tune_model(cat)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2530,0.0901,0.3002,0.0066,0.2054,1.1380
1,0.2514,0.0886,0.2977,0.0212,0.2055,1.0879
2,0.2399,0.0815,0.2854,0.0977,0.1968,1.0680
3,0.2387,0.0802,0.2832,0.1165,0.1948,1.0334
4,0.2377,0.0776,0.2786,0.1496,0.1909,1.0221
5,0.2258,0.0750,0.2738,0.1731,0.1884,1.0122
6,0.2295,0.0752,0.2743,0.1725,0.1860,0.9268
7,0.2401,0.0807,0.2840,0.1100,0.1932,1.0027
8,0.2235,0.0723,0.2689,0.2030,0.1837,0.9597


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [15]:
predict_model(cat_tune);
#predict_model(cat);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,CatBoost Regressor,0.2160,0.0679,0.2605,0.2433,0.1774,0.9473


In [17]:
newpred5 = predict_model(cat_tune, data=data_unseen)
#newpred5 = predict_model(cat, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,CatBoost Regressor,0.2264,0.0738,0.2717,0.1805,0.1845,0.9617


In [12]:
lgbm = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2757,0.1063,0.3261,-0.1716,0.2243,1.3451
1,0.2679,0.1033,0.3214,-0.1404,0.2221,1.1385
2,0.2551,0.0936,0.3060,-0.0369,0.2108,1.1376
3,0.2479,0.0903,0.3005,0.0053,0.2073,1.0517
4,0.2396,0.0823,0.2868,0.0989,0.1958,1.0079
5,0.2352,0.0836,0.2892,0.0774,0.1992,1.0131
6,0.2341,0.0811,0.2847,0.1085,0.1915,0.8319
7,0.2384,0.0846,0.2908,0.0669,0.1970,0.9364
8,0.2368,0.0801,0.2830,0.1172,0.1929,1.0323


In [13]:
lgbm_tune = tune_model(lgbm)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2486,0.0836,0.2891,0.0788,0.1988,1.1481
1,0.2458,0.0832,0.2885,0.0810,0.1993,1.0644
2,0.2403,0.0808,0.2843,0.1048,0.1953,1.0716
3,0.2382,0.0809,0.2844,0.1093,0.1951,1.0082
4,0.2334,0.0749,0.2737,0.1792,0.1874,1.0079
5,0.2292,0.0763,0.2762,0.1586,0.1908,1.0401
6,0.2279,0.0739,0.2719,0.1868,0.1846,0.9172
7,0.2411,0.0809,0.2844,0.1074,0.1942,1.0332
8,0.2223,0.0714,0.2673,0.2128,0.1823,0.9473


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [19]:
predict_model(lgbm_tune);
#predict_model(lgbm);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2142,0.0675,0.2598,0.2476,0.1760,0.9092


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [21]:
newpred5 = predict_model(lgbm_tune, data=data_unseen)
#newpred5 = predict_model(lgbm, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2256,0.0734,0.2709,0.1856,0.1837,0.9560


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [ ]:
blend1 = blend_models([rf_tune, lgbm_tune])

In [ ]:
blend1_tune = tune_model(blend1)

In [ ]:
#predict_model(blend1_tune);
predict_model(blend1);

In [ ]:
#newpred5 = predict_model(blend1_tune, data=data_unseen)
newpred5 = predict_model(blend1, data=data_unseen)

In [ ]:
blend2 = blend_models([rf_tune, lgbm_tune, cat_tune])

In [ ]:
blend2_tune = tune_model(blend2)

In [ ]:
predict_model(blend2_tune);
#predict_model(blend2);

In [ ]:
newpred6 = predict_model(blend2_tune, data=data_unseen)
#newpred6 = predict_model(blend2, data=data_unseen)

In [ ]:
blend3 = blend_models([rf_tune, lgbm_tune, gbr])

In [ ]:
blend3_tune = tune_model(blend3)

In [ ]:
predict_model(blend3_tune);
#predict_model(blend3);

In [ ]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

In [ ]:
blend4 = blend_models([rf_tune, lgbm_tune, gbr, cat_tune])

In [ ]:
blend4_tune = tune_model(blend4)

In [ ]:
#predict_model(blend4_tune);
predict_model(blend4);

In [ ]:
#newpred = predict_model(blend4_tune, data=data_unseen)
newpred = predict_model(blend4, data=data_unseen)

In [ ]:
blend5 = blend_models([rf_tune, cat_tune])

In [ ]:
blend5_tune = tune_model(blend5)

In [ ]:
predict_model(blend5_tune);
#predict_model(blend5);

In [ ]:
newpred = predict_model(blend5_tune, data=data_unseen)
#newpred = predict_model(blend5, data=data_unseen)

In [22]:
blend6 = blend_models([lgbm_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2491,0.0855,0.2924,0.0579,0.2005,1.1390
1,0.2480,0.0851,0.2918,0.0597,0.2016,1.0747
2,0.2392,0.0800,0.2829,0.1137,0.1947,1.0676
3,0.2382,0.0802,0.2832,0.1165,0.1946,1.0204
4,0.2347,0.0758,0.2754,0.1694,0.1886,1.0133
5,0.2269,0.0752,0.2742,0.1703,0.1892,1.0249
6,0.2284,0.0743,0.2726,0.1828,0.1850,0.9213
7,0.2405,0.0806,0.2838,0.1112,0.1934,1.0176
8,0.2225,0.0717,0.2677,0.2103,0.1827,0.9527


In [23]:
blend6_tune = tune_model(blend6)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2487,0.0841,0.2900,0.0733,0.1992,1.1438
1,0.2466,0.0839,0.2896,0.0737,0.2001,1.0686
2,0.2395,0.0802,0.2832,0.1118,0.1947,1.0690
3,0.2382,0.0805,0.2837,0.1134,0.1948,1.0134
4,0.2338,0.0752,0.2742,0.1761,0.1878,1.0099
5,0.2279,0.0757,0.2751,0.1648,0.1900,1.0328
6,0.2280,0.0740,0.2721,0.1858,0.1847,0.9188
7,0.2407,0.0807,0.2841,0.1097,0.1938,1.0261
8,0.2223,0.0715,0.2674,0.2123,0.1824,0.9495


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [25]:
predict_model(blend6_tune);
#predict_model(blend6);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2144,0.0674,0.2596,0.2485,0.1761,0.9172


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [ ]:
newpred = predict_model(blend6_tune, data=data_unseen)
#newpred = predict_model(blend6, data=data_unseen)

In [ ]:
blend7 = blend_models([lgbm_tune, gbr_tune])

In [ ]:
blend7_tune = tune_model(blend7)

In [ ]:
predict_model(blend7_tune);
#predict_model(blend7);

In [ ]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

In [ ]:
blend8 = blend_models([gbr_tune, cat_tune])

In [ ]:
blend8_tune = tune_model(blend8)

In [ ]:
predict_model(blend8_tune);
#predict_model(blend8);

In [ ]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

In [ ]:
save_model(blend6_tune, "indycar_rf_lgbm_gbr_prequaly_model_v2")

In [ ]:
save_model(lgbm_tune, "indycar_lgbm_postqualy_model_v4")